[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C40_Research_Methodology_Course/02_experiment_design/02_experiment_design.ipynb)

# 02 · 实验设计与消融纪律（动手做）

目标：把受控实验的原则跑成可验证的代码——**对照组**、**消融归因**、**识别并分离混杂**、**OFAT vs 因子设计**。

路线：无对照 vs 有对照 → 消融做归因 → 分离混杂(数据/算力) → 随机化与配对 → OFAT 漏掉交互 → ✏️ 练习 → 📖 答案 → 🧪 受控玩具实验胶囊。

> 核心心法：**对比的两个配置之间，只能差你声称的那一个变量。** 多差一个，归因就脏了。

## 1 · worked：没有对照，「涨了」毫无意义

我们造一个『真相 = A 与 B 完全一样好』的世界（真实差异=0），看『各跑一次比胜负』会得出什么荒谬结论，再看『各跑多次比均值±方差』如何还原真相。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

TRUE = 0.88        # A 和 B 真实水平完全相同
NOISE = 0.006      # 种子方差

def run(method, rng, n=1):
    return TRUE + rng.normal(0, NOISE, size=n)   # 与 method 无关：A、B 真的一样

# 无对照：各跑一次比胜负
single_wins = sum(run('B', rng)[0] > run('A', rng)[0] for _ in range(2000)) / 2000
print(f'真相：A 与 B 完全相同（差异=0）')
print(f'只各跑一次时 B「赢」的比例 = {single_wins:.0%}  <- 约抛硬币')
assert 0.4 < single_wins < 0.6

# 有对照：各跑 8 个种子，看区间是否重叠
a = run('A', rng, 8); b = run('B', rng, 8)
print(f'\nA: {a.mean():.4f} ± {a.std(ddof=1):.4f}')
print(f'B: {b.mean():.4f} ± {b.std(ddof=1):.4f}')
overlap = abs(a.mean() - b.mean()) < (a.std(ddof=1) + b.std(ddof=1))
assert overlap, '差异为0时，多种子的区间应重叠 -> 正确结论：无显著差异'
print('区间重叠 -> 正确结论：A 与 B 无显著差异（对照救了你）')
print('✅ 无对照=抛硬币定胜负；有对照+多种子=看出『其实没差异』。')

## 2 · worked：消融实验做贡献归因

方法 = 基线 + A + B + C。逐个移除组件，看掉多少分，把功劳归到组件，揪出名不副实的组件。

In [ ]:
ablation = pd.DataFrame({
    'config':   ['full(A+B+C)', '-A', '-B', '-C', 'baseline'],
    'accuracy': [0.910,         0.882, 0.905, 0.860, 0.840],
})
full = ablation.loc[ablation.config == 'full(A+B+C)', 'accuracy'].iloc[0]
base = ablation.loc[ablation.config == 'baseline', 'accuracy'].iloc[0]
contrib = {comp: full - ablation.loc[ablation.config == f'-{comp}', 'accuracy'].iloc[0]
           for comp in ['A','B','C']}
print('各组件边际贡献(去掉它掉的分)：')
for comp, v in sorted(contrib.items(), key=lambda kv: -kv[1]):
    print(f'  {comp}: {v:+.3f}')
print(f'\n所有组件总贡献(full - baseline) = {full - base:+.3f}')
print(f'各组件单独贡献之和           = {sum(contrib.values()):+.3f}')
print('两者不等 -> 组件间存在交互效应（非简单叠加）')
assert contrib['C'] > contrib['A'] > contrib['B']
assert contrib['B'] < 0.01, 'B 名不副实'
print('✅ 归因：C 主力、A 次之、B 几乎没用。若论文主打 B，则名实不符。')

## 3 · worked：分离混杂——把数据/算力的功劳从方法里摘出来

2×2 设计：是否加新模块 × 数据量(1x/2x)。论文爱用『裸基线 vs 全家桶』的不诚实对比。我们做受控分解。

In [ ]:
exp = pd.DataFrame({
    'module': [False, True,  False, True],
    'data':   ['1x', '1x',  '2x',  '2x'],
    'acc':    [0.840, 0.850, 0.862, 0.871],
})
def acc(m, d):
    return exp.loc[(exp.module==m)&(exp.data==d), 'acc'].iloc[0]

naive   = acc(True,'2x') - acc(False,'1x')    # 不诚实对比
pure_module = acc(True,'1x') - acc(False,'1x') # 控住数据，模块纯效应
pure_data   = acc(False,'2x') - acc(False,'1x')# 控住模块，数据纯效应
print(f'不诚实宣称的提升(裸基线 vs 全家桶) = {naive:+.3f}')
print(f'  模块纯效应(控住数据=1x)         = {pure_module:+.3f}')
print(f'  数据纯效应(控住无模块)          = {pure_data:+.3f}')
assert naive > pure_module, '混杂把宣称的提升放大了'
assert abs(naive - (pure_module + pure_data)) < 0.01, '总提升≈两个纯效应之和(本例近似可加)'
print('✅ 拆穿：宣称的+0.031里，模块只占+0.010，其余是数据(混杂)。')

## 4 · worked：随机化与配对——让对比更精

比较 A、B 时，让它们用**相同的一组种子**(配对设计)，种子带来的波动在两组间抵消，差异估计更精。

In [ ]:
def run_method(effect, seeds):
    '''每个种子产生一次运行；种子决定一个共同的随机波动 + 方法自身效应。'''
    out = []
    for s in seeds:
        r = np.random.default_rng(s)
        shared_noise = r.normal(0, 0.02)       # 该种子的『运气』，A/B 共享
        out.append(0.85 + effect + shared_noise + r.normal(0, 0.003))
    return np.array(out)

seeds = list(range(10))
a = run_method(0.000, seeds)      # 基线
b = run_method(0.010, seeds)      # 真实 +0.01

# 非配对：直接比均值差，方差大
unpaired_gap = b.mean() - a.mean()
# 配对：先逐种子相减，再看差值（共享噪声被消掉）
paired_diff = b - a
print(f'非配对均值差 = {unpaired_gap:+.4f}')
print(f'配对差值     = {paired_diff.mean():+.4f} ± {paired_diff.std(ddof=1):.4f}')
print(f'配对差值的标准差({paired_diff.std(ddof=1):.4f}) 远小于单组标准差({a.std(ddof=1):.4f})')
assert paired_diff.std(ddof=1) < a.std(ddof=1), '配对消掉共享噪声，差值方差更小'
assert abs(paired_diff.mean() - 0.010) < 0.005, '配对能精确还原真实 +0.01'
print('✅ 配对设计(共享种子)消掉共享波动，让真实 +0.01 清晰浮现。')

## 5 · worked：OFAT 漏掉交互效应

两个因素 A、B，各自单独无用，同时开却大涨。OFAT 会丢弃两者；因子设计(2×2)才能发现交互。

In [ ]:
factorial = pd.DataFrame({
    'A': [False, True, False, True],
    'B': [False, False, True, True],
    'acc': [0.840, 0.841, 0.842, 0.895],   # 只有 A&B 同时开才涨
})
def fa(a, b):
    return factorial.loc[(factorial.A==a)&(factorial.B==b), 'acc'].iloc[0]

main_A = fa(True, False) - fa(False, False)   # A 主效应(B关时)
main_B = fa(False, True) - fa(False, False)   # B 主效应(A关时)
both   = fa(True, True)  - fa(False, False)   # A&B 联合
interaction = both - main_A - main_B          # 交互 = 联合 - 两主效应
print(f'A 单独效应 = {main_A:+.3f}  (≈0，OFAT 会丢弃 A)')
print(f'B 单独效应 = {main_B:+.3f}  (≈0，OFAT 会丢弃 B)')
print(f'A&B 联合   = {both:+.3f}')
print(f'交互效应   = {interaction:+.3f}  <- OFAT 永远看不到！')
assert main_A < 0.01 and main_B < 0.01, '各自主效应≈0'
assert interaction > 0.04, '交互效应巨大'
print('✅ OFAT 会先后丢弃 A、B；因子设计(2×2)才揭示它们的强协同。')

---
## ✏️ 练习 1：检测混杂——对比是否干净？

给定两个实验配置（各是一个 dict，键是因素名、值是设置），实现 `clean_comparison(cfg_a, cfg_b, claimed_factor)`：
判断两配置**是否只在 `claimed_factor` 上不同**（其余全相同）。是 → True（干净对比）；否 → False（有混杂）。

In [ ]:
def clean_comparison(cfg_a, cfg_b, claimed_factor):
    # TODO: 找出两个配置中所有取值不同的键。
    #   若不同的键恰好只有 {claimed_factor} -> True，否则 False。
    #   提示：diff = {k for k in cfg_a if cfg_a[k] != cfg_b.get(k)}
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
base = dict(module=False, data='1x', lr=0.01, steps=1000)
clean = dict(module=True,  data='1x', lr=0.01, steps=1000)   # 只差 module
dirty = dict(module=True,  data='2x', lr=0.01, steps=1000)   # 差 module 和 data
assert clean_comparison(base, clean, 'module') == True,  '只差 module = 干净'
assert clean_comparison(base, dirty, 'module') == False, '还差了 data = 有混杂'
print('✅ 练习 1 通过：能判断一个对比是否只变了声称的那个变量')

## ✏️ 练习 2：构造消融矩阵

给定组件列表如 `['A','B']`，实现 `ablation_matrix(components)`：返回所有『开/关』组合的列表，
每个组合是一个 dict（键=组件名，值=True/False）。`n` 个组件应有 `2**n` 个组合（完整因子设计）。

In [ ]:
def ablation_matrix(components):
    # TODO: 返回所有 2**len(components) 种开关组合。
    #   提示：用 itertools.product([False, True], repeat=len(components))
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
m = ablation_matrix(['A', 'B'])
assert len(m) == 4, '2 个组件应有 4 种组合'
assert dict(A=False, B=False) in m and dict(A=True, B=True) in m
assert len(ablation_matrix(['A','B','C'])) == 8, '3 个组件 8 种'
print('✅ 练习 2 通过：能生成完整消融/因子矩阵')

## ✏️ 练习 3：随机化分组

把 `n` 个样本随机、**均衡**地分到 `k` 组（每组样本数尽量相等）。实现 `randomized_groups(n, k, seed)`：
返回长度 `n` 的数组，每个元素是该样本的组号(0..k-1)，且各组大小最多相差 1。

In [ ]:
def randomized_groups(n, k, seed=0):
    # TODO: 1) rng=np.random.default_rng(seed)
    #       2) 生成 [0,1,..,k-1,0,1,..] 循环到长度 n（保证均衡）
    #       3) rng.permutation 打乱后返回（随机化）
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
g = randomized_groups(100, 4, seed=1)
assert len(g) == 100
counts = np.bincount(g)
assert counts.max() - counts.min() <= 1, '各组大小最多相差1（均衡）'
assert set(g.tolist()) == {0,1,2,3}, '应恰好用到 k 个组'
# 不同种子应给出不同分配（随机化）
assert not np.array_equal(g, randomized_groups(100, 4, seed=2))
print('✅ 练习 3 通过：均衡 + 随机的分组')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def clean_comparison(cfg_a, cfg_b, claimed_factor):
    keys = set(cfg_a) | set(cfg_b)
    diff = {k for k in keys if cfg_a.get(k) != cfg_b.get(k)}
    return diff == {claimed_factor}

In [ ]:
# 练习 2 参考答案
import itertools
def ablation_matrix(components):
    out = []
    for combo in itertools.product([False, True], repeat=len(components)):
        out.append(dict(zip(components, combo)))
    return out

In [ ]:
# 练习 3 参考答案
def randomized_groups(n, k, seed=0):
    rng = np.random.default_rng(seed)
    base = np.array([i % k for i in range(n)])   # 均衡
    return rng.permutation(base)                 # 随机化

---
## 🧪 真实数据胶囊：在已知真相的沙盒里做完整受控实验

我们造一个**我们知道全部真相**的合成任务，埋下：组件 A 真有用(+0.05)、组件 B 无用(0)、数据量是混杂(+0.03)。
然后**假装不知道**，用受控实验把这些真相**测出来**，验证方法论确实能区分真有用 vs 假有用。

In [ ]:
def synthetic_run(use_A, use_B, data_2x, seed):
    '''返回一次运行的准确率。真相：A=+0.05, B=0, 数据2x=+0.03，叠加种子噪声。'''
    r = np.random.default_rng(seed)
    acc = 0.80
    if use_A:   acc += 0.05      # A 真有用
    if use_B:   acc += 0.00      # B 是装饰
    if data_2x: acc += 0.03      # 数据量(混杂)
    return acc + r.normal(0, 0.004)

SEEDS = list(range(12))
def mean_over_seeds(use_A, use_B, data_2x):
    return np.mean([synthetic_run(use_A, use_B, data_2x, s) for s in SEEDS])

# 受控测 A 的纯效应：固定 B=off、数据=1x，只切换 A
effect_A = mean_over_seeds(True, False, False) - mean_over_seeds(False, False, False)
# 受控测 B 的纯效应：固定 A=off、数据=1x，只切换 B
effect_B = mean_over_seeds(False, True, False) - mean_over_seeds(False, False, False)
# 不诚实对比：裸基线(1x) vs 全家桶(A+B+2x数据)
dishonest = mean_over_seeds(True, True, True) - mean_over_seeds(False, False, False)
print(f'受控测出 A 纯效应 = {effect_A:+.3f}  (真相 +0.050)')
print(f'受控测出 B 纯效应 = {effect_B:+.3f}  (真相  0.000)')
print(f'不诚实全家桶对比 = {dishonest:+.3f}  (虚增！含了数据混杂)')
assert abs(effect_A - 0.05) < 0.01, '受控实验应还原 A≈+0.05'
assert abs(effect_B - 0.00) < 0.01, '受控实验应识破 B≈0'
assert dishonest > effect_A + 0.02, '全家桶对比把真实效应虚增了(混杂)'
print('✅ 受控实验准确测出 A 真有用、B 无用，并识破全家桶对比的虚增。')

**🧪 胶囊练习**：实现 `interaction_AB()`：用 `mean_over_seeds` 测 A 与 B 的**交互效应**（数据固定 1x）。
= [A,B 都开] − [只 A] − [只 B] + [都关]。本沙盒里 B 无用，交互也应≈0。

In [ ]:
def interaction_AB():
    # TODO: 交互 = f(T,T) - f(T,F) - f(F,T) + f(F,F)，数据均为 1x（data_2x=False）
    #   f = lambda a,b: mean_over_seeds(a, b, False)
    raise NotImplementedError

In [ ]:
# 自测
inter = interaction_AB()
print(f'A×B 交互效应 = {inter:+.4f}  (本沙盒 B 无用，应≈0)')
assert abs(inter) < 0.01, '本沙盒无真实交互，应≈0'
print('✅ 胶囊练习通过：能用 2×2 估计交互效应')

In [ ]:
# 📖 胶囊参考答案
def interaction_AB():
    f = lambda a, b: mean_over_seeds(a, b, False)
    return f(True, True) - f(True, False) - f(False, True) + f(False, False)

### 小结
- **受控实验**：一次只变一个量，其余固定，才能把结果差异**归因**到那个量。
- **对照组**：没有对照，「涨了」无法区分真效应与本来的波动；对照要被公平对待(别立稻草人)。
- **消融**：逐个移除组件做归因；揪出名不副实的组件；警惕消融不重调参/不报方差。
- **混杂**：数据/算力/调参搭着方法偷偷增加 = 把它们的功劳记到方法头上；用**控制+随机化**消除。
- **OFAT vs 因子设计**：默认 OFAT(归因清晰)；怀疑有交互时用因子设计(能看到协同)。

下一站：**模块 03 · 研究统计** —— 有了受控实验的多次运行，怎么用统计判断差异是真信号还是噪声。